In [ ]:
# Import statements
import os
from git import Repo
import mimetypes
from tree_sitter_language_pack import get_parser

In [ ]:
# Convert text to UTF-8 bytes
def to_utf8_bytes(text):
    if isinstance(text, bytes):
        return text
    return text.encode("utf-8", errors="replace")

In [ ]:
# Clone repository
repo_url = "https://github.com/Inhuman-Jester/RAG-Project.git"
local_path = "./public"

# Repo.clone_from(repo_url, local_path)

repo = Repo(local_path)

In [6]:
print(repo.git.status())

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
# List all files in the repository
files = [
    item.path
    for item in repo.tree().traverse()
    if item.type == 'blob'
]
print(files)

['.gitignore',
 'Euthyphro.pdf',
 'app.py',
 'requirements.txt',
 'source/indexing_pipeline.py',
 'source/retrieval_and_generation_pipeline.py',
 'test/main.ipynb',
 'utils/config.py',
 'utils/helper.py']

In [8]:
# Read the content of a specific file
def read_file(repo, path):
    try:
        return repo.git.show(f'HEAD:{path}')
    except:
        return None

In [ ]:
# Process each file to get its content and type
code_files = []
for f in files:
    content = read_file(repo, f)
    _, ext = os.path.splitext(f)

    if(mimetypes.guess_type(f)[0]) : file_type=mimetypes.guess_type(f)[0].split('/')[-1] 
    else : file_type = ext[1:]
    
    if content:
        code_files.append({
            "path": f,
            "content": content,
            "file_type": file_type
        })


In [10]:

print(code_files[8]['file_type'])

x-python


In [ ]:
# Parse the content of the selected file
parser = get_parser("python")
code = code_files[8]['content']
tree = parser.parse(code.encode('utf8'))
root = tree.root_node

In [ ]:
# Walk the syntax tree to extract functions and classes
def walk(node, file, code):
    chunk = []

    if node.type == "function_definition":
        name = node.child_by_field_name("name").text.decode("utf-8")

        chunk.append({
            "id": name,
            "code": code[node.start_byte:node.end_byte].decode("utf-8", errors="ignore"),
            "metadata": {
                "type": "function",
                "language": file["file_type"],
                "file": file["path"],
                "dependencies": ["jwt"],
                "line_of_code": [
                    node.start_point[0] + 1,
                    node.end_point[0] + 1
                ]
            }
        })

    elif node.type == "class_definition":
        name = node.child_by_field_name("name").text.decode("utf-8")

        chunk.append({
            "id": name,
            "code": code[node.start_byte:node.end_byte].decode("utf-8", errors="ignore"),
            "metadata": {
                "type": "class",
                "language": file["file_type"],
                "file": file["path"],
                "dependencies": ["jwt"],
                "line_of_code": [
                    node.start_point[0] + 1,
                    node.end_point[0] + 1
                ]
            }
        })

    for child in node.children:
        chunk.extend(walk(child, file, code))

    return chunk


In [ ]:
# Extract chunks from all code files
chunks = []

for f in code_files:
    parser = get_parser("python")
    code_bytes = to_utf8_bytes(f["content"])

    tree = parser.parse(code_bytes)
    root = tree.root_node

    chunks.extend(walk(root, f, code_bytes))


In [32]:
(print(chunks))

[{'id': 'is_followup_question', 'code': 'def is_followup_question(user_query, context):\n    logging.info("1. Classifying the query as follow-up or not...")\n    # If the last Q&A pair is not available, return False\n    context = get_last_qa_context(st.session_state.chat_history)\n    if context == "":\n        return False\n\n    # Embedding-based follow-up detection using cosine similarity\n    # Since embeddings_model.embed_documents returns a list, we wrap them with np.array.\n    query_embedding = np.array(embeddings_model.encode([user_query]))\n    context_embedding = np.array(embeddings_model.encode([context]))\n\n    # cosine_similarity returns a 2D array (1x1 in this case), so we extract the first element.\n    similarity = cosine_similarity(query_embedding, context_embedding)[0][0]\n    logging.info(f"2. Follow-up Detection | Cosine Similarity: {\'True\' if similarity >= 0.25 else \'False\'} | Similarity: {similarity}")\n\n    # Rule-based follow-up detection\n    ambiguous_